# IMPORTS

In [144]:
from IPython.display import HTML, display
display(HTML('<style>.container { width:90% !important; }</style>'))
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
import pandas as pd
import openpyxl
from tkinter import simpledialog
import re
import shutil
import os
import requests
import json
from geopy.geocoders import Nominatim
from geopy import Point
from datetime import timedelta, datetime
from docxtpl import DocxTemplate
import subprocess
import pickle
import os.path
import secrets
from randomtimestamp import randomtimestamp, random_time
from random import randint
from docx import Document
from PyPDF2 import PdfMerger
from IPython.display import FileLink, FileLinks
from time import sleep
from geopy.exc import GeocoderTimedOut
import locale
from tqdm.notebook import tqdm, trange

    pd.get_option('display.max_rows')
    pd.get_option('display.max_columns')
    pd.get_option('display.max_colwidth')
    pd.get_option('display.expand_frame_repr')
    pd.set_option('display.expand_frame_repr', False)

    pd.set_option('display.max_rows', None)

# CONSTANTES

In [9]:
novembre = pd.read_csv('../application/data/csv/novembre.csv', sep=';')
novembre = novembre.sort_values(['date', 'time'])
novembre.reset_index(drop=True, inplace=True)
cdb = novembre.copy()

In [12]:
november = [x for x in novembre['date'].unique()]

In [145]:
june = [
    "28/06/23", "29/06/23", "30/06/23"
]

july = [
    "03/07/23", "04/07/23", "05/07/23", "06/07/23", "07/07/23",
    "10/07/23", "11/07/23", "12/07/23", "13/07/23",
    "17/07/23", "18/07/23", "19/07/23", "20/07/23", "21/07/23",
    "24/07/23", "25/07/23", "26/07/23", "27/07/23", "28/07/23"
]

august = [
    "28/08/23", "29/08/23", "30/08/23", "31/08/23"
]

september = [
    "01/09/23",
    "07/09/23", "08/09/23", 
    "11/09/23", "12/09/23", "13/09/23", "14/09/23", "15/09/23",
    "18/09/23", "19/09/23", "20/09/23", "21/09/23", "22/09/23",
    "25/09/23", "26/09/23", "27/09/23", "28/09/23", "29/09/23"
]

october = [
    "02/10/23", "03/10/23", "04/10/23", "05/10/23", "06/10/23",
    "09/10/23", "10/10/23", "11/10/23", "12/10/23", "13/10/23",
    "16/10/23", "17/10/23", "18/10/23", "19/10/23", "20/10/23",
    "23/10/23", "24/10/23", "25/10/23", "26/10/23", "27/10/23"
]

november = [
    '02/11/23', '03/11/23', 
    '06/11/23', '07/11/23', '08/11/23', '09/11/23', '10/11/23',
    '13/11/23', '14/11/23', '15/11/23', '16/11/23', '17/11/23',
    '20/11/23', '21/11/23', '22/11/23', '23/11/23', '24/11/23',
    '27/11/23', '28/11/23', '29/11/23', '30/11/23'
]

rootdir = 'novembre/pdfs/'
since = june + july + august + september + october + november

locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8');
geolocator = Nominatim(user_agent="mySecretAgent")

filepath="/home/julien/website/application/data/xls_x/carnet_de_bord.xlsx"
zones = pd.read_excel(filepath, sheet_name="zones")
zones.set_index("cp", inplace=True)

durations = [0.75, 1, 1.25, 1.5]

'fr_FR.UTF-8'

In [22]:
cdb.loc[82, 'date'] = "07/11/23"
cdb.loc[82, 'nom'] = "DANEL,Henri"
cdb.loc[82, 'time'] = "14:30"
cdb.loc[83, 'date'] = "07/11/23"
cdb.loc[83, 'nom'] = "LEBEL,Marie Amelie"
cdb.loc[83, 'time'] = "16:30"

In [23]:
cdb=cdb.sort_values(['date', 'time'])
cdb.reset_index(drop=True, inplace=True)

In [146]:
from app import app
app.app_context().push()

from application.dash.biocodex.models import Pharmacy, Identity, Cdb, Connections, Adress

from app import db

In [ ]:
for i, row in tqdm(cdb.iterrows(), total=len(cdb)):
    if "PHARM" not in row['nom']:
        doc = Identity.query.filter((Identity.nom==row['nom']) & (Identity.prenom==row['prenom'].title())).first()
        adr = Adress.query.filter(Adress.id == doc.connections[0].adr_id).first()
        cdb.loc[i, 'adr'] = adr.adr
        cdb.loc[i, 'zip'] = adr.cp
        cdb.loc[i, 'city'] = adr.ville
    else:
        pharmas = Pharmacy.query.filter(Pharmacy.ddv!=None).all()
        pharmas = [pharma.toDict() for pharma in pharmas]
        pharmas = pd.DataFrame(pharmas)
        pharma = pharmas[pharmas['nom']==row['nom']]
        cdb.loc[i, 'adr'] = pharma.adr.values[0]
        cdb.loc[i, 'zip'] = pharma.cp.values[0]
        cdb.loc[i, 'city'] = pharma.ville.values[0]

# GEOCODE

In [147]:
def do_geocode(address, attempt=1, max_attempts=5):
    try:
        return geolocator.geocode(address)
    except GeocoderTimedOut:
        if attempt <= max_attempts:
            return do_geocode(address, attempt=attempt+1)
        raise      

# DOCX TO PDF

In [148]:
def pdf_from_docx(input_docx, output_pathname):

    subprocess.call(
        [
            'soffice',
            # '--headless',
            '--convert-to',
            'pdf',
            '--outdir',
            output_pathname,
            input_docx
        ],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

    return None

# GET DAY

In [149]:
def get_day(date, month, dej="home"):
    
    cdb = pd.read_csv(f"/home/julien/website/application/data/csv/{month}.csv", sep=";")
    home_gps=geolocator.geocode("35 RUE MARCEL BONTEMPS 92100 BOULOGNE-BILLANCOURT")
    us_date = pd.to_datetime(date, dayfirst=True).date().strftime("%Y-%m-%d")
    
    home = {
        "type": "poi",        
        "date": date,
        "time": "07:30",
        "nom": "HOME",
        "adresse": "35 RUE MARCEL BONTEMPS",
        "cp": 92100,
        "ville": "BOULOGNE BILLANCOURT",
        "lat": home_gps.latitude,
        "lon": home_gps.longitude
    }
    lunch = {
        "type": "break",
        "date": date,
        "time": "13:15",
        "nom": "LUNCH HOME",
        "adresse": "35 RUE MARCEL BONTEMPS",
        "cp": 92100,
        "ville": "BOULOGNE BILLANCOURT",
        "lat": home_gps.latitude,
        "lon": home_gps.longitude  
    }
    diner = {
        "type": "poi",        
        "date": date,
        "time": "19:30",
        "nom": "BACK HOME",
        "adresse": "35 RUE MARCEL BONTEMPS",
        "cp": 92100,
        "ville": "BOULOGNE BILLANCOURT",
        "lat": home_gps.latitude,
        "lon": home_gps.longitude  
    }

    if dej=="home":
        df = pd.DataFrame([home, lunch, diner])
    else:
        df = pd.DataFrame([home, diner])
    
    df["lat"] = pd.Series(dtype=float, index=df.index)
    df["lon"] = pd.Series(dtype=float, index=df.index)
    
    df.columns=[
        "type", "date", "time", "id", "adr", "zip", "city", "lat", "lon" 
    ]
    
    for i, row in df.iterrows():
        local = geolocator.geocode(f"{row['adr']} {str(int(row['zip']))} {row['city']}")
        df.loc[i, "lat"] = local.latitude
        df.loc[i, "lon"] = local.longitude
        
    tournee = cdb[["date", "time",  "nom", "adr", "zip", "city", "lat", "lon"]][cdb["date"]==date].copy()
    print("CONTACTS:\t", len(tournee))
    tournee['half'] = pd.Series(dtype=str)
    
    for i, row in tournee.iterrows():

        if pd.to_datetime(row['time']).time() <= pd.to_datetime("13:00").time():
            tournee.loc[i, 'half'] = 'am'
        else:
            tournee.loc[i, 'half'] = 'pm'
        
        tournee.loc[i, "type"] = "call"
    
    if dej != "home":

        lasts = tournee[tournee['half']=="am"]
        last = lasts[lasts['time']==lasts['time'].max()]
        #print("LAST\n", last)

        if len(tournee[tournee['half']=="pm"]) > 0:
            firsts = tournee[tournee['half']=="pm"]
            first = firsts[firsts['time']==firsts['time'].min()]
            #print("FIRST\n", first)
            dej_lat=(last['lat'].values[0]+first['lat'].values[0])/2
            dej_lon=(last['lon'].values[0]+first['lon'].values[0])/2 
            tournee.loc[i+1, 'date'] = date
            tournee.loc[i+1, 'time'] = "13:15"
            tournee.loc[i+1, "type"] = "dej"
            tournee.loc[i+1, "id"] = "DEJ"
            tournee.loc[i+1, "lat"] = dej_lat
            tournee.loc[i+1, "lon"] = dej_lon
            adresse = geolocator.reverse(Point(dej_lat, dej_lon)).raw['address']
            tournee.loc[i+1, 'zip'] = adresse.get('postcode')
            tournee.loc[i+1, 'city'] = adresse.get('town') or adresse.get('city')
            tournee.loc[i+1, 'city'] = tournee.loc[i+1, 'city'].upper()
            tournee.loc[i+1, 'adr'] = adresse.get('road').upper()


        else:
            
            local = geolocator.geocode(f"{lunch['adresse']} {lunch['cp']} {lunch['ville']}")
            dej_lat = local.latitude
            dej_lon = local.longitude
            tournee.loc[i+1, 'date'] = date
            tournee.loc[i+1, 'time'] = "13:15"
            tournee.loc[i+1, "type"] = "dej"
            tournee.loc[i+1, "id"] = "DEJ"
            tournee.loc[i+1, "lat"] = dej_lat
            tournee.loc[i+1, "lon"] = dej_lon
            adresse = geolocator.reverse(Point(dej_lat, dej_lon)).raw['address']
            tournee.loc[i+1, 'zip'] = adresse.get('postcode')
            tournee.loc[i+1, 'city'] = adresse.get('town') or adresse.get('city')
            tournee.loc[i+1, 'city'] = tournee.loc[i+1, 'city'].upper()
            tournee.loc[i+1, 'adr'] = adresse.get('road').upper()

            
    day = pd.concat([tournee, df], ignore_index=True)       
    # day["ddv"] = pd.to_datetime(day["ddv"], format="mixed")
    day = day.sort_values(by=["date", "time"]).fillna("")
    day.reset_index(drop=True, inplace=True)
    day["type"] = day["type"].replace("", "call")
    
    return day

# RANDOM DEPARTURE

In [150]:
def get_random_time(date, h_min="08:30", h_max="09:30"):

    t_min = datetime.strptime(f"{h_min}", "%H:%M").time()
    t_max = datetime.strptime(f"{h_max}", "%H:%M").time()
    rand_t = random_time(start=t_min, end=t_max, pattern="%H:%M")
    rand_time = rand_t.strftime("%H:%M")

    return pd.to_datetime(f"{date} {rand_time}", format="%d/%m/%y %H:%M")

# ITINARIES & CAR PARK

    datetime.strptime(date, "%d/%m/%y").date().strftime("%d %B %Y")

In [ ]:
bar_fmt = "{n_fmt}/{total_fmt} {desc} {bar} {percentage:3.0f}% elapsed:{elapsed} remaining:{remaining}"

In [151]:
def get_itinaries_and_parks(date: str, day: pd.DataFrame):
    
    departure_d_t = get_random_time(date)
    
    ts = pd.to_datetime(date, dayfirst=True)
    filename = ts.date().strftime("%Y%m%d")
    
    trajets = pd.DataFrame(columns = ["from", "to", "distance", "departure", "duration", "arrival"])
    parkings = pd.DataFrame(columns=["pbp_date", "park", "start", "stay", "stop", "code_zone", "nom_zone", "ville", "price"])
    
    for i, row in tqdm(day.iloc[:-1].iterrows(), total=len(day)-1, desc="trajets", leave=False):
        
        origin = day.loc[i]
        lat_0 = row["lat"]
        lon_0 = row["lon"]    
        trajets.loc[i, "from"] = origin["nom"]

        destination = day.loc[i+1]
        lat_1 = day.loc[i+1, "lat"]
        lon_1 = day.loc[i+1, "lon"]
        trajets.loc[i, "to"] = destination["nom"]
        
        print(origin['nom'])
        print(destination['zip'])

        trajets.loc[i, "departure"] = departure_d_t.time()

        r = requests.get(f"http://router.project-osrm.org/route/v1/car/{lon_0},{lat_0};{lon_1},{lat_1}?overview=false""")
        sleep(1)

        routes = json.loads(r.content)
        route = routes.get("routes")[0]

        duration = int(round(route["duration"]/60,0))
        trajets.loc[i, "duration"] = duration
        
        arrival_d_t = datetime.combine(departure_d_t.date(), departure_d_t.time()) + timedelta(minutes = duration)
        trajets.loc[i, "arrival"] = arrival_d_t.time()
        
        distance = round(route['distance']/1000, 1)    
        trajets.loc[i, "distance"] = distance
        
        
        if destination["type"] == "call" or destination["type"] == "dej":            
            
            start = arrival_d_t + timedelta(minutes=9)
            
            parkings.loc[i, "pbp_date"] = datetime.strptime(date, "%d/%m/%y").date().strftime("%d %B %Y")
            parkings.loc[i, "park"] = f'PARK_{destination["id"]}'            
            parkings.loc[i, "start"] = start.time().strftime("%H:%M")
            parkings.loc[i, "stay"] = durations[randint(0,3)]
            
            stop = start + timedelta(minutes=parkings.loc[i, "stay"]*60)
            
            parkings.loc[i, "stop"] = stop.time().strftime("%H:%M")
            code_zone = destination["zip"]
            parkings.loc[i, "code_zone"] = code_zone
            parkings.loc[i, "nom_zone"] = zones.loc[int(code_zone), "nom zone"]
            parkings.loc[i, "pbp_ville"] = zones.loc[int(code_zone), "ville"]
            price = (parkings.loc[i, "stay"] * zones.loc[int(code_zone), "tarif"]) + 0.25
            parkings.loc[i, "price"] = price
            
            departure_d_t = stop + timedelta(minutes=8)
            
            
        elif destination["type"] == "break":
            
            start = arrival_d_t + timedelta(minutes=9)            
            break_duration = 45
            stop = start + timedelta(minutes=break_duration)
            departure_d_t = stop + timedelta(minutes=8)
            
            
    return trajets, parkings

In [ ]:
t, p = get_itinaries_and_parks(date="02/11/23", day=day)
p

# GET INVOICES

In [ ]:
l_bar=’{desc}: {percentage:3.0f}%|’ 
r_bar=’| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, ‘ ‘{rate_fmt}{postfix}]’ 

In [152]:
def get_pbp_invoices(date: str, pdfs=True):
    
    if date in ["14/11/23", "22/11/23", "27/11/23"]:
        day = get_day(date, 'novembre', dej="home")  
    else:
        day = get_day(date, 'novembre', dej="else")
        
    trajets, parkings = get_itinaries_and_parks(date, day)
    parkings.reset_index(drop=True, inplace=True)
    parkings["price"] = parkings["price"].replace(".", ",", regex=False)
    ts = pd.to_datetime(date, dayfirst=True)
    filename = ts.date().strftime("%Y%m%d")
    
    prix_total = "{:.2f}".format(parkings["price"].sum())
    dist_total = int(round(trajets["distance"].sum(), 0))    
    month = ts.date().strftime("%B")
    
    common_path = f"{month}"
            
    for j, parking in tqdm(parkings.iterrows(), total=len(parkings), desc="pdfs", leave=False):

        park = {
            "date_2": date,
            "date": parking["pbp_date"],
            "start": parking["start"],
            "end": parking["stop"],
            "ville": parking["pbp_ville"],
            "nom": parking["nom_zone"],
            "code": parking["code_zone"],
            "price": "{:.2f}".format(parking["price"]).replace(".", ",")
        }
        
        if pdfs:
            page = DocxTemplate("/home/julien/website/frais/pbp_tpl.docx")
            page.render(park)
            path_docx = f"{common_path}/docx/pbp_{filename}_{j+1}.docx"
            page.save(path_docx)
            pdf_from_docx(path_docx, f"{common_path}/pdfs/")
            os.remove(path_docx)
                
    return prix_total, dist_total, j+1

  # EXECUTE

In [122]:
execute = {}

In [153]:
def process_date(date, pdfs=True):
    
    print(date)
    
    prix_total, dist_total, n = get_pbp_invoices(date, pdfs)
    
    print(f"n_kms: \t\t{dist_total}\nparks:\t\t{prix_total}\nn_pdfs:\t\t{n}\n")

    date = datetime.strptime(date, "%d/%m/%y").date().strftime("%Y%m%d")
    
    execute[date] = {
        "total parkings" : prix_total,
        "total kms": dist_total,
        "filepaths": [f"novembre/pdfs/pbp_{date}_{x+1}.pdf" for x in range(n)]
    }
    
    return execute

    for date in june:
        execute = process_date(date)

    for date in july:
        execute = process_date(date)

    for date in octo:
        execute = process_date(date)

In [ ]:
for date in november:
    execute = process_date(date, pdfs=False)

In [154]:
execute = process_date("30/11/23", pdfs=False)

30/11/23
CONTACTS:	 4


trajets:   0%|          | 0/6 [00:00<?, ?it/s]


75015.0
CHEDEVERGNE,Frederique
75015.0
LAMBE,Cecile
75006

75015.0
LAGEIX,Florence
75015.0
LARDEUX,Claude
92100


pdfs:   0%|          | 0/5 [00:00<?, ?it/s]

n_kms: 		16
parks:		25.25
n_pdfs:		5



In [155]:
resume = pd.DataFrame.from_dict(execute, orient="index")

In [156]:
resume.index = pd.to_datetime(resume.index, format="%Y%m%d").strftime('%d/%m/%y')

In [157]:
def make_clickable(url_list):    
    names = [os.path.basename(url) for url in url_list]    
    return [f"<a href='{url}'>{name}</a>" for url, name in zip(url_list, names)]

In [158]:
resume.style.format({"filepaths": make_clickable})

,total parkings,total kms,filepaths
02/11/23,28.25,22,"[""pbp_20231102_1.pdf"", ""pbp_20231102_2.pdf"", ""pbp_20231102_3.pdf"", ""pbp_20231102_4.pdf"", ""pbp_20231102_5.pdf""]"
03/11/23,25.50,18,"[""pbp_20231103_1.pdf"", ""pbp_20231103_2.pdf"", ""pbp_20231103_3.pdf"", ""pbp_20231103_4.pdf"", ""pbp_20231103_5.pdf"", ""pbp_20231103_6.pdf""]"
06/11/23,21.25,8,"[""pbp_20231106_1.pdf"", ""pbp_20231106_2.pdf"", ""pbp_20231106_3.pdf"", ""pbp_20231106_4.pdf"", ""pbp_20231106_5.pdf""]"
07/11/23,21.75,20,"[""pbp_20231107_1.pdf"", ""pbp_20231107_2.pdf"", ""pbp_20231107_3.pdf"", ""pbp_20231107_4.pdf"", ""pbp_20231107_5.pdf""]"
08/11/23,27.50,22,"[""pbp_20231108_1.pdf"", ""pbp_20231108_2.pdf"", ""pbp_20231108_3.pdf"", ""pbp_20231108_4.pdf"", ""pbp_20231108_5.pdf"", ""pbp_20231108_6.pdf""]"
09/11/23,19.00,14,"[""pbp_20231109_1.pdf"", ""pbp_20231109_2.pdf"", ""pbp_20231109_3.pdf"", ""pbp_20231109_4.pdf""]"
10/11/23,21.25,14,"[""pbp_20231110_1.pdf"", ""pbp_20231110_2.pdf"", ""pbp_20231110_3.pdf"", ""pbp_20231110_4.pdf"", ""pbp_20231110_5.pdf""]"
13/11/23,24.25,14,"[""pbp_20231113_1.pdf"", ""pbp_20231113_2.pdf"", ""pbp_20231113_3.pdf"", ""pbp_20231113_4.pdf"", ""pbp_20231113_5.pdf""]"
14/11/23,11.75,21,"[""pbp_20231114_1.pdf"", ""pbp_20231114_2.pdf"", ""pbp_20231114_3.pdf""]"
15/11/23,21.50,26,"[""pbp_20231115_1.pdf"", ""pbp_20231115_2.pdf"", ""pbp_20231115_3.pdf"", ""pbp_20231115_4.pdf""]"


In [ ]:
resume['total parkings'] = resume['total parkings'].astype(float)
resume['total parkings'].sum()

In [ ]:
resume

In [ ]:
resume.style.format({'filepath': make_clickable})

# UPDATE XLSX BOOK

In [ ]:
def update_calls_sheet(date, filepath=filepath):
    
    calls_of_the_day = get_calls_by_date(date)
    print(f"nb of calls: {calls_of_the_day.shape[0]-3}")
        
    with pd.ExcelWriter(
        filepath, engine='openpyxl', if_sheet_exists="replace", mode="a", date_format="dd/mm/yy", datetime_format="HH:MM"
    ) as writer:

        if "calls" not in writer.sheets:
            
            calls_of_the_day.to_excel(writer, sheet_name="calls", index=False)  
            print("Creating sheet 'calls'")
            # print("Calls: sheet updated")
            
        else:
            
            existing_calls = pd.read_excel(filepath, sheet_name="calls", engine='openpyxl')

            dates = set([row["ddv"].date() for i, row in existing_calls.iterrows()])
            
            ts = pd.to_datetime(date, format="%d/%m/%y")

            if ts.date() in dates:

                print("date already in calls")
                # print("no need to update sheet calls")

            else:

                concat = pd.concat([existing_calls, calls_of_the_day])
                
                concat.to_excel(writer, sheet_name="calls", index=False)
                
                # print("Calls: sheet updated")

    return None 

In [ ]:
def update_kms_sheet(day_steps, filepath=filepath):

    date = day_steps[0]
    dist_tot_park_tot_dict = {}
    dist_tot_park_tot_dict[date] = {}
    dist_tot_park_tot_dict[date]["kms"] = day_steps[-2]
    dist_tot_park_tot_dict[date]["parkings"] = day_steps[-1]
    
    kms_of_the_day = pd.DataFrame.from_dict(dist_tot_park_tot_dict, orient="index")
    kms_of_the_day.reset_index(drop=False, inplace=True)
    kms_of_the_day.columns=["date", "kms pro", "total parking"]
            
    with pd.ExcelWriter(
        filepath, engine='openpyxl', if_sheet_exists="replace", mode="a", date_format="dd/mm/yy", datetime_format="HH:MM"
    ) as writer:    
        
        if "kms" not in writer.sheets:
            
            kms_of_the_day.to_excel(writer, sheet_name="kms", index=False)  
            print("Creating sheet 'kms'")
            # print("Kms: sheet updated")
            
        else:

            kmsdf = pd.read_excel(filepath, sheet_name="kms", engine='openpyxl')

            dates = set([row["date"] for i, row in kmsdf.iterrows()])

            if date in dates:

                print("date already in kms")
                # print("no need to update sheet kms")

            else:
                
                concat = pd.concat([kmsdf, kms_of_the_day])
                concat.to_excel(writer, sheet_name="kms", index=False)

                # print("Kilometers: sheet updated")
                                
    return None

In [ ]:
day_steps = get_day_steps("28/06/23")

In [ ]:
day_steps[-1]

In [ ]:
def update_sheets(date, filepath=filepath):

    update_calls_sheet(date)
    update_kms_sheet(date)

    return None